# Normalisierung einer Datenbank mit Python und SQLite
## Ziel des Notebooks

In diesem Notebook wird schrittweise gezeigt, wie eine relationale Datenbank 
- analysiert,
- normalisiert (1NF → 2NF → 3NF),
- und anschließend wieder als neues Datenbankschema gespeichert wird.

Dabei werden folgende Technologien verwendet:
- Python (insb. Pandas)
- SQLite und SQL

### Wichtiger Hinweis

Für die Ausführung des Notebooks wird die Datei: *Chicken.db* benötigt.
Die Datenbankdatei muss sich im selben Verzeichnis wie das Notebook befinden.

# Imports
Es werden die benötigten Python-Bibliotheken importiert:
- sqlite3 für die Verbindung zur SQLite-Datenbank
- pandas für die Verarbeitung tabellarischer Daten

In [1]:
import sqlite3
import pandas as pd

# Verbindung zur Datenbank aufbauen
Es wird eine Verbindung zur SQLite-Datenbank aufgebaut. Zusätzlich wird ein sogenannter Cursor erzeugt, mit dem SQL-Abfragen ausgeführt werden können.

In [2]:
conn = sqlite3.connect("Chicken.db")
cursor = conn.cursor()

# Überblick über die Ausgangsdaten
Zunächst betrachten wir die vorhandenen Relationen der Datenbank, um die Struktur der Daten besser zu verstehen.

### Relation ARTIKELÜBERSICHT:

In [3]:
query =("""
SELECT *
FROM ARTIKELUEBERSICHT;
""")

result = cursor.execute(query).fetchall()
result

[(1, 'Turbo Huhn', 'Huhn', 59.95),
 (2, 'Premium Huhn', 'Huhn', 129.95),
 (3, 'Bio Huhn', 'Huhn', 179.95),
 (4, 'Hühnerfutter', 'Futter', 29.95)]

### Reation BESTELLÜBERSICHT:

In [4]:
query =("""
SELECT *
FROM BESTELLUEBERSICHT;
""")

result = cursor.execute(query).fetchall()
result

[(1,
  '14.03.2026',
  1,
  'Pauline Poulet',
  'Huhndertwasserweg 17, 04109 Eipzig',
  '3',
  '1',
  '179.95',
  '0.2'),
 (2,
  '03.05.2026',
  2,
  'Patricio Pollo',
  'Hahnenbusch 1, 69115 Eidelberg',
  '2',
  '2',
  '129.95',
  '0.2'),
 (3,
  '19.04.2026',
  3,
  'Jacky Chickenwing',
  'Huhnsrückstraße 10, 30159 Hahnover',
  '1, 3, 4',
  '1, 1, 2',
  '59.95, 179.95, 29.95',
  '0.2,  0.2,  0.0'),
 (4,
  '23.05.2026',
  1,
  'Pauline Poulet',
  'Huhndertwasserweg 17, 04109 Eipzig',
  '4',
  '1',
  '29.95',
  '0.0'),
 (5,
  '01.06.2026',
  4,
  'Hildegard Huhn',
  'Hahns-Thoma Weg 123, 63450 Hahnau',
  '1',
  '3',
  '59.95',
  ' 0.2')]

# Laden der Daten in 1NF
Die Überführung in die 1. Normalform (1NF) wird in diesem Notebook nicht näher betrachtet und ist Teil der Übungsaufgabe.
Für dieses Beispiel liegen die Daten bereits in 1NF in der Datenbank als *GESAMTSICHT_1NF* vor.

Zur besseren Weiterverarbeitung werden die Daten nun in einen Pandas-DataFrame geladen.

In [5]:
gesamtsicht_1nf_df = pd.read_sql_query("""
SELECT *    
FROM GESAMTSICHT_1NF
""", conn)

gesamtsicht_1nf_df

,Bestell_ID,Bestelldatum,Kunden_ID,Vorname,Nachname,Straße,Hausnummer,PLZ,Stadt,Artikel_ID,Artikelbezeichnung,Artikelgruppe,Menge,Preis,Rabatt
0,1,14.03.2026,1,Pauline,Poulet,Huhndertwasserweg,17,04109,Eipzig,3,Bio Huhn,Huhn,1,179.95,0.2
1,2,03.05.2026,2,Patricio,Pollo,Hahnenbusch,1,69115,Eidelberg,2,Premium Huhn,Huhn,2,129.95,0.2
2,3,19.04.2026,3,Jacky,Chickenwing,Huhnsrückstraße,10,30159,Hahnover,1,Turbo Huhn,Huhn,1,59.95,0.2
3,3,19.04.2026,3,Jacky,Chickenwing,Huhnsrückstraße,10,30159,Hahnover,3,Bio Huhn,Huhn,1,179.95,0.2
4,3,19.04.2026,3,Jacky,Chickenwing,Huhnsrückstraße,10,30159,Hahnover,4,Hühnerfutter,Futter,2,29.95,0.0
5,4,23.05.2026,1,Pauline,Poulet,Huhndertwasserweg,17,04109,Eipzig,4,Hühnerfutter,Futter,1,29.95,0.0
6,5,01.06.2026,4,Hildegard,Huhn,Hahns-Thoma Weg,123,63450,Hahnau,1,Turbo Huhn,Huhn,3,59.95,0.2


# Überführen in 2NF:

Die 2NF entfernt:
- partielle funktionale Abhängigkeiten
- redundante Daten

Wir prüfen beispielhaft, ob bestimmte Werte mehrfach vorkommen:

In [6]:
gesamtsicht_1nf_df["Straße"].value_counts()

Straße
Huhnsrückstraße      3
Huhndertwasserweg    2
Hahnenbusch          1
Hahns-Thoma Weg      1
Name: count, dtype: int64

Z.B Straßennamen treten mehrfach auf. Dies deutet auf Redundanzen innerhalb der Relation hin.

Wie in der Vorlesung gesehen, können wir die Daten in drei Relationen aufteilen, sodass kein Nichtschlüsselattribut von einem Teil eines Kandidatenschlüssels funktional abhängt:

#### Relation ARTIKEL:
Alle artikelbezogenen Informationen werden in eine eigene Relation ausgelagert. Jeder Artikel erscheint nur noch einmal.

In [7]:
artikel_df = gesamtsicht_1nf_df[["Artikel_ID", "Artikelbezeichnung", "Artikelgruppe", "Preis", "Rabatt"]].drop_duplicates()
artikel_df

,Artikel_ID,Artikelbezeichnung,Artikelgruppe,Preis,Rabatt
0,3,Bio Huhn,Huhn,179.95,0.2
1,2,Premium Huhn,Huhn,129.95,0.2
2,1,Turbo Huhn,Huhn,59.95,0.2
4,4,Hühnerfutter,Futter,29.95,0.0


#### Relation BESTELLUNG:
Alle bestellbezogenen Informationen werden in eine eigene Relation ausgelagert. Jede Bestellung erscheint nur noch einmal.

In [8]:
bestellung_df = gesamtsicht_1nf_df[["Bestell_ID", "Bestelldatum","Kunden_ID","Vorname", "Nachname", "Straße", "Hausnummer", "PLZ", "Stadt"]].drop_duplicates()
bestellung_df

,Bestell_ID,Bestelldatum,Kunden_ID,Vorname,Nachname,Straße,Hausnummer,PLZ,Stadt
0,1,14.03.2026,1,Pauline,Poulet,Huhndertwasserweg,17,04109,Eipzig
1,2,03.05.2026,2,Patricio,Pollo,Hahnenbusch,1,69115,Eidelberg
2,3,19.04.2026,3,Jacky,Chickenwing,Huhnsrückstraße,10,30159,Hahnover
5,4,23.05.2026,1,Pauline,Poulet,Huhndertwasserweg,17,04109,Eipzig
6,5,01.06.2026,4,Hildegard,Huhn,Hahns-Thoma Weg,123,63450,Hahnau


#### Relation BESTELLPOSITION:
Die Beziehung zwischen Bestellungen und Artikeln wird separat gespeichert.

In [9]:
bestellposition_df = gesamtsicht_1nf_df[["Bestell_ID","Artikel_ID", "Menge"]].drop_duplicates()
bestellposition_df

,Bestell_ID,Artikel_ID,Menge
0,1,3,1
1,2,2,2
2,3,1,1
3,3,3,1
4,3,4,2
5,4,4,1
6,5,1,3


# Überführen in 3NF:
Die 3NF entfernt:
- transitive funktionale Abhängigkeiten

In unserem Beispiel hängt der Rabatt von der Produktgruppe ab. Es ergbit sich also die transitive Abhängigkeit *Artikel_ID* → *Artikelgruppe* → *Rabatt*. Daher muss die Relation ARTIKEL in zwei Relationen zerlegt werden: 

#### Relation ARTIKELGRUPPE:
Der Zusammenhang zwischen Artikelgruppe und Rabatt wird in eine eigene Relation ausgelagert:

In [10]:
artikelgruppe_df = artikel_df[["Artikelgruppe", "Rabatt"]].drop_duplicates()
artikelgruppe_df

,Artikelgruppe,Rabatt
0,Huhn,0.2
4,Futter,0.0


#### Relation ARTIKEL (jetzt ohne Attribut *Rabatt*):
Die Relation ARTIKEL enthält nun nur noch direkt vom Primärschlüssel abhängige Attribute:

In [11]:
artikel_df = artikel_df[["Artikel_ID", "Artikelbezeichnung", "Artikelgruppe", "Preis"]].drop_duplicates()
artikel_df

,Artikel_ID,Artikelbezeichnung,Artikelgruppe,Preis
0,3,Bio Huhn,Huhn,179.95
1,2,Premium Huhn,Huhn,129.95
2,1,Turbo Huhn,Huhn,59.95
4,4,Hühnerfutter,Futter,29.95


Zudem gilt: *Bestell_ID* → *Kunden_ID* → *Vorname*, *Nachname*, *Straße*, *Hausnummer*, *PLZ*, *Stadt*. Die Kundeninformationen hängen also nicht direkt von der Bestellung ab. Daher muss auch die Relation BESTELLUNG noch in zwei Relationen zerlegt werden: 

#### Relation KUNDE:
Kundendaten werden in eine eigene Relation ausgelagert.

In [12]:
kunde_df = bestellung_df[["Kunden_ID", "Vorname", "Nachname", "Straße", "Hausnummer", "PLZ", "Stadt"]].drop_duplicates()
kunde_df

,Kunden_ID,Vorname,Nachname,Straße,Hausnummer,PLZ,Stadt
0,1,Pauline,Poulet,Huhndertwasserweg,17,04109,Eipzig
1,2,Patricio,Pollo,Hahnenbusch,1,69115,Eidelberg
2,3,Jacky,Chickenwing,Huhnsrückstraße,10,30159,Hahnover
6,4,Hildegard,Huhn,Hahns-Thoma Weg,123,63450,Hahnau


#### Relation BESTELLUNG (jetzt ohne Kundeninformationen):
Die Relation BESTELLUNG enthält nun nur noch bestellungsbezogene Informationen.

In [13]:
bestellung_df = bestellung_df[["Bestell_ID", "Bestelldatum","Kunden_ID"]].drop_duplicates()
bestellung_df

,Bestell_ID,Bestelldatum,Kunden_ID
0,1,14.03.2026,1
1,2,03.05.2026,2
2,3,19.04.2026,3
5,4,23.05.2026,1
6,5,01.06.2026,4


# Neues normalisiertes Datenbankschema erzeugen:
Die normalisierten Relationen werden nun in einer neuen SQLite-Datenbank "*Chicken_normalisiert.db*" gespeichert:

In [14]:
conn = sqlite3.connect("Chicken_normalisiert.db")
cursor = conn.cursor()

### Relation KUNDE:
Tabelle KUNDE anlegen:

In [15]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS KUNDE (
    Kunden_ID INTEGER INTEGER PRIMARY KEY,
    Vorname VARCHAR(50),
    Nachname VARCHAR(50),
    Straße VARCHAR(50),
    Hausnummer VARCHAR(50),
    PLZ VARCHAR(50),
    Stadt VARCHAR(50))
""")

Daten in die Tabelle schreiben:

In [16]:
kunde_df.to_sql("KUNDE", conn, if_exists="replace", index=False)

4

### Relation BESTELLUNG:
Tabelle BESTELLUNG anlegen:

In [17]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS BESTELLUNG (
    Bestell_ID INTEGER PRIMARY KEY,
    Bestelldatum DATE,
    Kunden_ID INTEGER
    )
""")


Daten in die Tabelle schreiben:

In [18]:
bestellung_df.to_sql("BESTELLUNG", conn, if_exists="replace", index=False)

5

### Relation ARTIKEL:
Tabelle ARTIKEL anlegen:

In [19]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS ARTIKEL (
    Artikel_ID INTEGER PRIMARY KEY,
    Artikelbezeichnung VARCHAR(50),
    Artikelgruppe VARCHAR(50),
    Preis REAL
    )
""")

Daten in die Tabelle schreiben:

In [20]:
artikel_df.to_sql("ARTIKEL", conn, if_exists="replace", index=False)

4

### Relation ARTIKELGRUPPE:
Tabelle ARTIKELGRUPPE anlegen:

In [21]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS ARTIKELGRUPPE (
    Artikelgruppe VARCHAR(50)  PRIMARY KEY,
    Rabatt REAL
    )
""")

Daten in die Tabelle schreiben:

In [22]:
artikelgruppe_df.to_sql("ARTIKELGRUPPE", conn, if_exists="replace", index=False)

2

### Relation BESTELLPOSITION:
Tabelle BESTELLPOSITION anlegen:

In [23]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS ARTIKELGRUPPE (
    Bestell_ID INTEGER,
    Artikel_ID INTEGER,
    Menge INTEGER,

   PRIMARY KEY (Bestell_ID, Artikel_ID)
    )
""")

Daten in die Tabelle schreiben:

In [24]:
bestellposition_df.to_sql("BESTELLPOSITION", conn, if_exists="replace", index=False)

7

# Beispielabfragen (SQL)
Nachdem die Datenbank erfolgreich normalisiert wurde, können die neuen Relationen nun mithilfe von SQL-Abfragen ausgewertet werden.

Die folgenden Beispiele zeigen typische Operationen auf relationalen Datenbanken:

#### Alle Artikel

In [25]:
query =("""
SELECT *
FROM ARTIKEL;
""")

result = cursor.execute(query).fetchall()
result

[(3, 'Bio Huhn', 'Huhn', 179.95),
 (2, 'Premium Huhn', 'Huhn', 129.95),
 (1, 'Turbo Huhn', 'Huhn', 59.95),
 (4, 'Hühnerfutter', 'Futter', 29.95)]

#### Die Mengen und Artikelbezeichnungen aus Bestellung 3:

In [26]:
query =("""
SELECT
    BP.Menge,
    A.Artikelbezeichnung
FROM BESTELLPOSITION BP
JOIN ARTIKEL A
    ON BP.Artikel_ID = A.Artikel_ID
JOIN BESTELLUNG B
    ON BP.Bestell_ID = B.Bestell_ID
WHERE B.Bestell_ID = 3;
""")

result = cursor.execute(query).fetchall()
result

[(1, 'Turbo Huhn'), (1, 'Bio Huhn'), (2, 'Hühnerfutter')]

#### Der Umsatz pro Kunde (inkl. Rabatt):

In [27]:
query =("""
SELECT
    K.Kunden_ID,
    K.Vorname,
    K.Nachname,
    SUM(BP.Menge * A.Preis * (1-AG.Rabatt)) AS Umsatz
FROM KUNDE K
JOIN BESTELLUNG B
    ON K.Kunden_ID = B.Kunden_ID
JOIN BESTELLPOSITION BP
    ON B.Bestell_ID = BP.Bestell_ID
JOIN ARTIKEL A
    ON BP.Artikel_ID = A.Artikel_ID
Join ARTIKELGRUPPE AG
    ON A.Artikelgruppe = AG.Artikelgruppe
GROUP BY
    K.Kunden_ID
ORDER BY Umsatz DESC;
""")

result = cursor.execute(query).fetchall()
result

[(3, 'Jacky', 'Chickenwing', 251.82000000000002),
 (2, 'Patricio', 'Pollo', 207.92),
 (1, 'Pauline', 'Poulet', 173.91),
 (4, 'Hildegard', 'Huhn', 143.88000000000002)]